
<div dir="ltr" align="center">

  <span style="font-size: 60px; color: #0B3C78; font-weight: 700;">
    Artificial Intelligence
  </span><br><br>

  <span style="font-size: 22px; color: #1F5FA8; font-weight: 500;">
    Computer Engineering Department
  </span><br>

  <span style="font-size: 20px; color: #1F5FA8; font-weight: 500;">
    Sharif University of Technology
  </span><br><br>

  <span style="font-size: 22px; color: #1F7A8C; font-weight: 600;">
    Spring 2026
  </span><br><br>

  <span style="font-size: 24px; color: #145DA0; font-weight: 700;">
    Practical Assignment
  </span><br><br>

  <span style="font-size: 22px; color: #3A7CC2; font-weight: 600;">
    Checkers
  </span><br><br><br>

</div>

---

- Instructor: Dr.Tanghatari
- Practical Designer: MohammadAli Meschi

---

### Student Information

- Name: Arvin
- Last Name: Baghal Asl
- Student Number: 403105793

---

### Instructions

- Make sure to:
  - Fill in your personal information above.
  - Run all the cells in the notebook and make sure your code runs truly

In [1]:
! pip install pygame


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# **Checkers**

task:
- Implement **Minimax**, **Alpha-Beta Pruning**, and **Expectimax** to make agents professional in Checkers.
- Ensure **the AI follows the rules of Checkers** and plays optimally.  

---

## **🛠️ Game Rules**

## Basic Setup
- Played on an **8×8 board** (same as chess).
- Each player starts with **12 pieces**.
- Pieces are placed on the **dark squares only**.
- Players sit opposite each other.

## Objective
- Capture all opponent’s pieces **or**
- Block them so they **cannot make a move**.

## Movement
- Pieces move **diagonally** on dark squares.
- Regular pieces can only move **forward** (toward the opponent).
- Move is typically **one square diagonally forward**.

## Capturing (Jumping)
- If an opponent’s piece is diagonally adjacent and the square beyond it is empty:
  - You **must jump over it** and capture it.
- Captured pieces are **removed from the board**.
- If multiple jumps are possible, you must continue jumping (**multi-capture**).

## Mandatory Captures
- If a capture is available, the player **must take it**.
- You cannot choose a normal move if a capture exists.

## Kinging
- When a piece reaches the opponent’s **back row**, it becomes a **king**.
- It is usually marked by stacking another piece on top.

## King Movement
- Kings can move **diagonally forward and backward**.
- Kings can also capture in **both directions**.
- In standard American checkers, kings move **one square at a time**.

## Turns
- Players alternate turns.
- Only **one piece is moved per turn** (except in multi-captures).

## Winning the Game
- You win if:
  - Your opponent has **no pieces left**, or
  - Your opponent **cannot make a legal move**.

## Draw Conditions
- The game may end in a draw if:
  - Moves are repeated continuously.
  - Neither player can force a win.

  ---

## **Step 1: Load Required Libraries**
Below is the game logic there is no need to change anything.

In [2]:
# checkers.py
import pygame

# Constants
WIDTH, HEIGHT = 600, 600
GRID_SIZE = 8
CELL_SIZE = WIDTH // GRID_SIZE
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GREEN = (34, 139, 34)
GRAY = (200, 200, 200)
BROWN = (139, 69, 19)
LIGHT_BROWN = (205, 133, 63)
RED = (255, 0, 0)
BLUE = (0, 0, 255)

class Checkers:
    def __init__(self):
        self.board = [[' ' for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
        self.initialize_board()
        self.current_player = 'B'  # B for Black (Human), W for White (AI)
        self.turn_count = 1
        self.must_jump = False
        self.valid_jump_moves = []
        self.waiting_for_second_jump = False
        self.jumping_piece = None
        
    def initialize_board(self):
        # Place black pieces (rows 0-2)
        for row in range(3):
            for col in range(GRID_SIZE):
                if (row + col) % 2 != 0:  # Only dark squares
                    self.board[row][col] = 'B'
        
        # Place white pieces (rows 5-7)
        for row in range(5, 8):
            for col in range(GRID_SIZE):
                if (row + col) % 2 != 0:  # Only dark squares
                    self.board[row][col] = 'W'
    
    def is_king(self, piece):
        """Check if a piece is a king"""
        return len(piece) == 2 and piece[1] == 'K'
    
    def has_any_jump(self):
        """Check if current player has any jump moves available"""
        for row in range(GRID_SIZE):
            for col in range(GRID_SIZE):
                piece = self.board[row][col]
                if piece != ' ' and piece[0] == self.current_player:
                    if self.get_all_jumps(row, col):
                        return True
        return False
    
    def is_valid_move(self, from_row, from_col, to_row, to_col):
        """Check if a move is valid - MUST consider forced jumps"""
        # Check if target is within bounds
        if not (0 <= to_row < GRID_SIZE and 0 <= to_col < GRID_SIZE):
            return False
        
        # Check if source has a piece
        piece = self.board[from_row][from_col]
        if piece == ' ':
            return False
        
        # Check if target is empty
        if self.board[to_row][to_col] != ' ':
            return False
        
        # Check if it's the current player's piece
        if piece[0] != self.current_player:
            return False
        
        # CRITICAL FIX: If there are any jumps available, only jumps are allowed
        if self.has_any_jump():
            # Check if this move is a jump
            is_jump = abs(to_row - from_row) == 2
            if not is_jump:
                return False  # Regular moves not allowed when jumps exist
        
        # Calculate differences
        row_diff = to_row - from_row
        col_diff = abs(to_col - from_col)
        
        # Check for valid move pattern
        is_king_piece = self.is_king(piece)
        
        # Regular move (one step diagonal)
        if abs(row_diff) == 1 and col_diff == 1:
            # Regular pieces can only move forward
            if not is_king_piece:
                # Black moves down (positive row), White moves up (negative row)
                if self.current_player == 'B' and row_diff == 1:
                    return True
                elif self.current_player == 'W' and row_diff == -1:
                    return True
                else:
                    return False
            else:
                # Kings can move in any diagonal direction
                return True
        
        # Jump move (two steps diagonal)
        elif abs(row_diff) == 2 and col_diff == 2:
            # Check if jumping over an opponent piece
            mid_row = (from_row + to_row) // 2
            mid_col = (from_col + to_col) // 2
            mid_piece = self.board[mid_row][mid_col]
            
            # Must have an opponent piece to capture
            if mid_piece == ' ' or mid_piece[0] == self.current_player:
                return False
            
            # Regular pieces can only jump forward
            if not is_king_piece:
                # Black jumps down (positive row), White jumps up (negative row)
                if self.current_player == 'B' and row_diff == 2:
                    return True
                elif self.current_player == 'W' and row_diff == -2:
                    return True
                else:
                    return False
            else:
                # Kings can jump in any diagonal direction
                return True
        
        return False
    
    def get_all_jumps(self, row, col):
        """Get all possible jump moves from a given position"""
        jumps = []
        piece = self.board[row][col]
        if piece == ' ':
            return jumps
        
        is_king_piece = self.is_king(piece)
        directions = []
        
        # Determine possible directions
        if is_king_piece:
            # Kings can move in all 4 diagonal directions
            directions = [(2, 2), (2, -2), (-2, 2), (-2, -2)]
        else:
            # Regular pieces can only move forward
            if piece[0] == 'B':
                directions = [(2, 2), (2, -2)]  # Down-right, Down-left
            else:  # 'W'
                directions = [(-2, 2), (-2, -2)]  # Up-right, Up-left
        
        for dr, dc in directions:
            target_row = row + dr
            target_col = col + dc
            mid_row = row + dr // 2
            mid_col = col + dc // 2
            
            # Check if jump is valid
            if (0 <= target_row < GRID_SIZE and 0 <= target_col < GRID_SIZE and
                self.board[target_row][target_col] == ' ' and
                self.board[mid_row][mid_col] != ' ' and
                self.board[mid_row][mid_col][0] != self.current_player):
                jumps.append((target_row, target_col, mid_row, mid_col))
        
        return jumps
    
    def get_all_piece_jumps(self):
        """Get all possible jump moves for the current player"""
        all_jumps = []
        for row in range(GRID_SIZE):
            for col in range(GRID_SIZE):
                piece = self.board[row][col]
                if piece != ' ' and piece[0] == self.current_player:
                    jumps = self.get_all_jumps(row, col)
                    for jump in jumps:
                        all_jumps.append((row, col, jump[0], jump[1], jump[2], jump[3]))
        return all_jumps
    
    def get_valid_moves_for_piece(self, row, col):
        """Get all valid moves (regular and jumps) for a specific piece"""
        if self.board[row][col] == ' ' or self.board[row][col][0] != self.current_player:
            return []
        
        # Check if there are any jumps available in the game
        if self.has_any_jump():
            # Return only jumps for this piece
            jumps = self.get_all_jumps(row, col)
            return [(jump[0], jump[1]) for jump in jumps]
        
        # Otherwise return regular moves
        moves = []
        piece = self.board[row][col]
        is_king_piece = self.is_king(piece)
        
        # Determine regular move directions (1 step)
        if is_king_piece:
            directions = [(1, 1), (1, -1), (-1, 1), (-1, -1)]
        else:
            if piece[0] == 'B':
                directions = [(1, 1), (1, -1)]  # Down-right, Down-left
            else:  # 'W'
                directions = [(-1, 1), (-1, -1)]  # Up-right, Up-left
        
        for dr, dc in directions:
            target_row = row + dr
            target_col = col + dc
            if (0 <= target_row < GRID_SIZE and 0 <= target_col < GRID_SIZE and
                self.board[target_row][target_col] == ' '):
                moves.append((target_row, target_col))
        
        return moves
    
    def get_valid_moves(self):
        """Get all valid moves for the current player"""
        # First check if any jumps are available
        if self.has_any_jump():
            # Return only jump moves
            all_jumps = self.get_all_piece_jumps()
            return [(jump[0], jump[1], jump[2], jump[3]) for jump in all_jumps]
        
        # Otherwise return regular moves
        moves = []
        for row in range(GRID_SIZE):
            for col in range(GRID_SIZE):
                piece_moves = self.get_valid_moves_for_piece(row, col)
                for target_row, target_col in piece_moves:
                    moves.append((row, col, target_row, target_col))
        return moves
    
    def make_move(self, from_row, from_col, to_row, to_col):
        """Make a move on the board"""
        if not self.is_valid_move(from_row, from_col, to_row, to_col):
            return False
        
        # Check if it's a jump move
        is_jump = abs(to_row - from_row) == 2
        
        # Move the piece
        piece = self.board[from_row][from_col]
        self.board[to_row][to_col] = piece
        self.board[from_row][from_col] = ' '
        
        # Handle capture
        if is_jump:
            mid_row = (from_row + to_row) // 2
            mid_col = (from_col + to_col) // 2
            self.board[mid_row][mid_col] = ' '
            
            # Check if this piece can make another jump
            additional_jumps = self.get_all_jumps(to_row, to_col)
            if additional_jumps:
                # Store that we need to continue jumping with this piece
                self.must_jump = True
                self.waiting_for_second_jump = True
                self.jumping_piece = (to_row, to_col)
                self.valid_jump_moves = [(to_row, to_col, jump[0], jump[1]) for jump in additional_jumps]
                return True
            else:
                # No more jumps, clear jump state
                self.must_jump = False
                self.waiting_for_second_jump = False
                self.jumping_piece = None
                self.valid_jump_moves = []
        
        # Check for king promotion
        if piece[0] == 'B' and to_row == GRID_SIZE - 1:
            self.board[to_row][to_col] = 'BK'
        elif piece[0] == 'W' and to_row == 0:
            self.board[to_row][to_col] = 'WK'
        
        # Switch players
        if not self.must_jump:
            self.current_player = 'B' if self.current_player == 'W' else 'W'
            self.turn_count += 1
        
        return True
    
    def forced_jump_move(self, to_row, to_col):
        """Complete a forced jump move (used for multiple jumps)"""
        if not self.must_jump or not self.valid_jump_moves:
            return False
        
        # Find the matching jump
        for jump in self.valid_jump_moves:
            if jump[2] == to_row and jump[3] == to_col:
                from_row, from_col = jump[0], jump[1]
                # Perform the jump
                piece = self.board[from_row][from_col]
                self.board[to_row][to_col] = piece
                self.board[from_row][from_col] = ' '
                
                # Remove captured piece
                mid_row = (from_row + to_row) // 2
                mid_col = (from_col + to_col) // 2
                self.board[mid_row][mid_col] = ' '
                
                # Check for king promotion
                if piece[0] == 'B' and to_row == GRID_SIZE - 1:
                    self.board[to_row][to_col] = 'BK'
                elif piece[0] == 'W' and to_row == 0:
                    self.board[to_row][to_col] = 'WK'
                
                # Check for additional jumps
                additional_jumps = self.get_all_jumps(to_row, to_col)
                if additional_jumps:
                    # Continue jumping with same piece
                    self.jumping_piece = (to_row, to_col)
                    self.valid_jump_moves = [(to_row, to_col, jump[0], jump[1]) for jump in additional_jumps]
                    return True
                else:
                    # No more jumps, switch players
                    self.must_jump = False
                    self.waiting_for_second_jump = False
                    self.jumping_piece = None
                    self.valid_jump_moves = []
                    self.current_player = 'B' if self.current_player == 'W' else 'W'
                    self.turn_count += 1
                    return True
        
        return False
    
    def game_over(self):
        """Check if the game is over"""
        # Check if current player has any moves
        return len(self.get_valid_moves()) == 0
    
    def get_winner(self):
        """Get the winner"""
        black_count = sum(row.count('B') + row.count('BK') for row in self.board)
        white_count = sum(row.count('W') + row.count('WK') for row in self.board)
        
        if black_count == 0:
            return "AI Wins!"  # White wins
        elif white_count == 0:
            return "You Win!"  # Black wins
        else:
            # No moves left, count pieces
            if black_count > white_count:
                return "You Win!"
            elif white_count > black_count:
                return "AI Wins!"
            else:
                return "It's a Draw!"

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## **Step 2: Implement AI Agents**
TODO: Implement Minimax Agent
Minimax recursively chooses the best move by:

Maximizing its own advantage.
Minimizing the opponent's advantage.
Exploring a tree of possible game states.

In [3]:
# minimax_agent.py
import copy

class MinimaxAgent:
	def __init__(self, depth, ai_color='W'):
		self.depth = depth
		self.ai_color = ai_color

	def get_move(self, game: Checkers):
		""" Runs Minimax to determine the best move. """
		best_move = None
		best_score = float('-inf')
		
		if not game.get_valid_moves():
			return None
		
		# Get all valid moves
		valid_moves = game.get_valid_moves()
		
		for move in valid_moves:
			# Create a deep copy of the game state
			temp_game = copy.deepcopy(game)
			
			# Make the move
			temp_game.make_move(move[0], move[1], move[2], move[3])
			
			# Evaluate the move from the opponent's perspective (minimizing)
			score = self.minimax(temp_game, self.depth - 1, False)
			
			# Track the best move for maximizing player (White/AI)
			if score > best_score:
				best_score = score
				best_move = move
		
		return best_move
	
	def minimax(self, game: Checkers, depth, maximizing_player):
		""" Recursively explores possible moves using Minimax. """
		# Base case: reached depth limit or game over
		if depth == 0 or game.game_over():
			return self.evaluate(game)
		
		moves = game.get_valid_moves()

		if maximizing_player:
			max_eval = float('-inf')

			for move in moves:
				child = copy.deepcopy(game)
				child.make_move(move[0], move[1], move[2], move[3])
				eval = self.minimax(child, depth - 1, child.current_player == 'W')
				max_eval = max(max_eval, eval)

			return max_eval
		else:
			min_eval = float('inf')

			for move in moves:
				child = copy.deepcopy(game)
				child.make_move(move[0], move[1], move[2], move[3])
				eval = self.minimax(child, depth - 1, child.current_player == 'W')
				min_eval = min(min_eval, eval)
			
			return min_eval
	
	def evaluate(self, game: Checkers):
		""" Evaluates the board state for White (AI) vs Black (Human). """
		white_score = 0
		black_score = 0
		
		for row in range(8):
			for col in range(8):
				piece = game.board[row][col]
				if piece == ' ':
					continue

				# Material value
				if piece == 'W':
					value = 1
					# Positional bonus: closer to row 0 (promotion) is better
					value += (7 - row) * 0.1
				elif piece == 'WK':
					value = 3
				elif piece == 'B':
					value = 1
					# Black's promotion row is row 7, so closer to 7 is better
					value += row * 0.1
				elif piece == 'BK':
					value = 3
				else:
					value = 0

				if piece[0] == 'W':
					white_score += value
				elif piece[0] == 'B':
					black_score += value

		diff = white_score - black_score
		return diff if self.ai_color == 'W' else -diff


📝 TODO: Implement Alpha-Beta Pruning
Alpha-Beta Pruning optimizes Minimax by eliminating unnecessary branches.

In [4]:
# alpha_beta_agent.py
import copy

class AlphaBetaAgent:
	def __init__(self, depth, ai_color='W'):
		self.depth = depth
		self.ai_color = ai_color

	def get_move(self, game: Checkers):
		""" Runs Alpha-Beta Pruning to determine the best move for White (AI). """
		best_move = None
		best_score = float('-inf')
		alpha = float('-inf')
		beta = float('inf')

		valid_moves = game.get_valid_moves()
		if not valid_moves:
			return None
		
		# Sort moves to try jumps first (better pruning)
		valid_moves.sort(key=lambda m: abs(m[2] - m[0]) == 2, reverse=True)
		
		for move in valid_moves:
			# Create a deep copy of the game state
			temp_game = copy.deepcopy(game)
			
			# Make the move (move format: from_row, from_col, to_row, to_col)
			success = temp_game.make_move(move[0], move[1], move[2], move[3])
			
			if success:
				# Handle multiple jumps
				while temp_game.must_jump and temp_game.waiting_for_second_jump:
					jump_moves = temp_game.valid_jump_moves
					if jump_moves:
						# Take the first available jump (in evaluation, we explore all)
						jump = jump_moves[0]
						temp_game.forced_jump_move(jump[2], jump[3])
					else:
						break
				
				# Evaluate the move
				score = self.alpha_beta(temp_game, self.depth - 1, alpha, beta, False)
				
				if score > best_score:
					best_score = score
					best_move = move

				# Alpha-Beta pruning
				if best_score >= beta:
					return best_move
				alpha = max(alpha, best_score)
		
		return best_move
	
	def alpha_beta(self, game: Checkers, depth, alpha, beta, maximizing_player):
		""" Applies pruning to Minimax for efficiency. """
		# Base case: reached depth limit or game over
		if depth == 0 or game.game_over():
			return self.evaluate(game)
		
		moves = game.get_valid_moves()

		if maximizing_player:
			max_eval = float('-inf')

			for move in moves:
				child = copy.deepcopy(game)
				child.make_move(move[0], move[1], move[2], move[3])
				eval = self.alpha_beta(child, depth - 1, alpha, beta, child.current_player == 'W')

				alpha = max(alpha, eval)
				if alpha >= beta:
					return alpha

				max_eval = max(max_eval, eval)

			return max_eval
		else:
			min_eval = float('inf')

			for move in moves:
				child = copy.deepcopy(game)
				child.make_move(move[0], move[1], move[2], move[3])
				eval = self.alpha_beta(child, depth - 1, alpha, beta, child.current_player == 'W')

				beta = min(beta, eval)
				if alpha >= beta:
					return beta

				min_eval = min(min_eval, eval)

			return min_eval
	
	def evaluate(self, game: Checkers):
		""" Advanced evaluation for Checkers from White's (AI) perspective. """
		white_score = 0
		black_score = 0
		
		# Piece values
		REGULAR_VALUE = 10
		KING_VALUE = 30
		
		for row in range(8):
			for col in range(8):
				piece = game.board[row][col]
				if piece == ' ':
					continue
				# Material
				if piece == 'W':
					score = REGULAR_VALUE + (7 - row) * 1   # advance bonus
				elif piece == 'WK':
					score = KING_VALUE
				elif piece == 'B':
					score = REGULAR_VALUE + row * 1
				elif piece == 'BK':
					score = KING_VALUE
				else:
					score = 0
				if piece[0] == 'W':
					white_score += score
				else:
					black_score += score
		
		# Mobility bonus (having more moves is generally good)
		# Save current player, temporarily set to White to check White's moves
		current_player = game.current_player
		game.current_player = 'W'
		white_moves = len(game.get_valid_moves())
		game.current_player = 'B'
		black_moves = len(game.get_valid_moves())
		game.current_player = current_player
		
		white_score += white_moves * 0.3
		black_score += black_moves * 0.3
		
		# Return score from White's perspective (positive = good for AI)
		diff = white_score - black_score
		return diff if self.ai_color == 'W' else -diff

📝 TODO: Implement Expectimax Agent
Expectimax is used when the opponent’s move is not optimal and follows a probabilistic strategy.

In [5]:
# expectimax_agent_simple.py
import copy

class ExpectimaxAgent:
	def __init__(self, depth, ai_color='W'):
		self.depth = depth
		self.ai_color = ai_color

	def get_move(self, game: Checkers):
		""" Runs Expectimax to determine the best move for White (AI). """
		best_move = None
		best_score = float('-inf')

		valid_moves = game.get_valid_moves()
		if not valid_moves:
			return None
		
		# Sort moves to try jumps first
		valid_moves.sort(key=lambda m: abs(m[2] - m[0]) == 2, reverse=True)
		
		for move in valid_moves:
			temp_game = copy.deepcopy(game)
			success = temp_game.make_move(move[0], move[1], move[2], move[3])
			
			if success:
				# Handle multiple jumps
				while temp_game.must_jump and temp_game.waiting_for_second_jump:
					jump_moves = temp_game.valid_jump_moves
					if jump_moves:
						jump = jump_moves[0]
						temp_game.forced_jump_move(jump[2], jump[3])
					else:
						break
				
				score = self.expectimax(temp_game, self.depth - 1, False)
				
				if score > best_score:
					best_score = score
					best_move = move
		
		return best_move
	
	def expectimax(self, game: Checkers, depth, maximizing_player):
		""" Uses probability-weighted decision-making instead of Minimax. """
		# Base case
		if depth == 0 or game.game_over():
			return self.evaluate(game)
		
		valid_moves = game.get_valid_moves()
		if not valid_moves:
			return self.evaluate(game)

		moves = game.get_valid_moves()

		if maximizing_player:
			max_eval = float('-inf')

			for move in moves:
				child = copy.deepcopy(game)
				child.make_move(move[0], move[1], move[2], move[3])
				eval = self.expectimax(child, depth - 1, child.current_player == 'W')
				max_eval = max(max_eval, eval)

			return max_eval
		else:
			total = 0

			for move in moves:
				child = copy.deepcopy(game)
				child.make_move(move[0], move[1], move[2], move[3])
				eval = self.expectimax(child, depth - 1, child.current_player == 'W')
				total += eval
			
			return total/len(moves)
	
	def evaluate(self, game: Checkers):
		""" Evaluates the board state based on piece count with king values. """
		white_score = 0
		black_score = 0
		
		for row in range(8):
			for col in range(8):
				piece = game.board[row][col]
				if piece == ' ':
					continue

				# Material value
				if piece == 'W':
					value = 1
				elif piece == 'WK':
					value = 3
				elif piece == 'B':
					value = 1
				elif piece == 'BK':
					value = 3
				else:
					value = 0

				if piece[0] == 'W':
					white_score += value
				elif piece[0] == 'B':
					black_score += value
		
		diff = white_score - black_score
		return diff if self.ai_color == 'W' else -diff

You can use the code below to initiate an agent and play with yourself

In [6]:
# main.py
import pygame

# Initialize Pygame
pygame.init()
screen = pygame.display.set_mode((600, 600))
pygame.display.set_caption("Checkers - Human vs AI")

# Load game instance
game = Checkers()

# Choose AI agent (Modify as needed)
# agent = MinimaxAgent(depth=4)
# agent = AlphaBetaAgent(depth=6)
agent = ExpectimaxAgent(depth=4)

# Colors
LIGHT_BROWN = (205, 133, 63)
DARK_BROWN = (139, 69, 19)
BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
GRAY = (200, 200, 200)
BLUE = (0, 0, 255)
RED = (255, 0, 0)
YELLOW = (255, 255, 0)

def draw_board(screen, game, selected_piece=None, valid_moves=None, forced_jump_piece=None):
    """ Draws the Checkers board and pieces """
    # Draw the board
    for row in range(8):
        for col in range(8):
            if (row + col) % 2 == 0:
                color = LIGHT_BROWN
            else:
                color = DARK_BROWN
            pygame.draw.rect(screen, color, (col * 75, row * 75, 75, 75))
    
    # Highlight selected piece
    if selected_piece:
        row, col = selected_piece
        pygame.draw.rect(screen, RED, (col * 75, row * 75, 75, 75), 3)
        
        # Show valid moves for selected piece
        if valid_moves:
            for move_row, move_col in valid_moves:
                pygame.draw.circle(screen, BLUE, 
                                 (move_col * 75 + 37, move_row * 75 + 37), 15)
    
    # Highlight forced jump piece
    if forced_jump_piece and game.must_jump:
        row, col = forced_jump_piece
        pygame.draw.rect(screen, YELLOW, (col * 75, row * 75, 75, 75), 4)
    
    # Draw pieces
    for row in range(8):
        for col in range(8):
            piece = game.board[row][col]
            if piece != ' ':
                center_x = col * 75 + 37
                center_y = row * 75 + 37
                
                # Draw piece
                if piece[0] == 'B':
                    color = BLACK
                else:
                    color = WHITE
                pygame.draw.circle(screen, color, (center_x, center_y), 30)
                pygame.draw.circle(screen, GRAY, (center_x, center_y), 30, 2)
                
                # Draw crown for kings
                if len(piece) == 2 and piece[1] == 'K':
                    crown_color = (255, 215, 0)  # Gold
                    pygame.draw.polygon(screen, crown_color, [
                        (center_x - 12, center_y - 8),
                        (center_x - 6, center_y - 14),
                        (center_x, center_y - 8),
                        (center_x + 6, center_y - 14),
                        (center_x + 12, center_y - 8)
                    ])
    
    # Display game info
    font = pygame.font.Font(None, 36)
    small_font = pygame.font.Font(None, 24)
    
    # Turn count
    turn_text = font.render(f"Turn: {game.turn_count}", True, WHITE)
    screen.blit(turn_text, (10, 10))
    
    # Current player
    if game.current_player == 'B':
        player_text = "Your Turn (Black)"
        player_color = BLACK
    else:
        player_text = "AI Turn (White)"
        player_color = WHITE
    
    player_surface = font.render(player_text, True, player_color)
    screen.blit(player_surface, (10, 50))
    
    # Forced jump indicator
    if game.must_jump:
        jump_text = small_font.render("MUST JUMP!", True, RED)
        screen.blit(jump_text, (10, 90))
    
    # Piece count
    black_count = sum(row.count('B') + row.count('BK') for row in game.board)
    white_count = sum(row.count('W') + row.count('WK') for row in game.board)
    count_text = small_font.render(f"Your Pieces: {black_count}  AI Pieces: {white_count}", True, WHITE)
    screen.blit(count_text, (10, 115))
    
    # AI difficulty
    depth_text = small_font.render(f"AI Depth: {agent.depth}", True, GRAY)
    screen.blit(depth_text, (10, 140))
    
    pygame.display.flip()

def show_game_over_screen(screen, game, winner_text):
    """Display game over screen"""
    overlay = pygame.Surface((600, 600))
    overlay.set_alpha(200)
    overlay.fill((0, 0, 0))
    screen.blit(overlay, (0, 0))
    
    font_large = pygame.font.Font(None, 72)
    font_small = pygame.font.Font(None, 36)
    
    # Winner text
    winner_surface = font_large.render(winner_text, True, YELLOW)
    winner_rect = winner_surface.get_rect(center=(300, 250))
    screen.blit(winner_surface, winner_rect)
    
    # Final score
    black_count = sum(row.count('B') + row.count('BK') for row in game.board)
    white_count = sum(row.count('W') + row.count('WK') for row in game.board)
    score_text = font_small.render(f"Final Score - You: {black_count}  AI: {white_count}", True, WHITE)
    score_rect = score_text.get_rect(center=(300, 350))
    screen.blit(score_text, score_rect)
    
    # Restart instruction
    restart_text = font_small.render("Press R to restart or ESC to quit", True, GRAY)
    restart_rect = restart_text.get_rect(center=(300, 450))
    screen.blit(restart_text, restart_rect)
    
    pygame.display.flip()

def ai_move(game, agent):
    """Execute AI move and return True if move was made"""
    ai_move_coords = agent.get_move(game)
    if ai_move_coords:
        if len(ai_move_coords) == 4:
            game.make_move(ai_move_coords[0], ai_move_coords[1], 
                          ai_move_coords[2], ai_move_coords[3])
            return True
    return False

# Game variables
running = True
game_over = False
winner_text = ""
selected_piece = None
ai_thinking = False  # Flag to prevent multiple AI calls
ai_move_timer = 0    # Timer to delay AI move

# Game loop
while running:
    if not game_over:
        # Draw board
        valid_moves = None
        forced_jump_piece = None
        
        if selected_piece and game.current_player == 'B':  # Only show moves on human turn
            row, col = selected_piece
            # Check if this piece can make a mandatory jump
            if game.must_jump:
                # Only show moves for pieces that can jump
                for jump in game.valid_jump_moves:
                    if jump[0] == row and jump[1] == col:
                        valid_moves = [(jump[2], jump[3])]
                        forced_jump_piece = (row, col)
                        break
            else:
                valid_moves = game.get_valid_moves_for_piece(row, col)
        
        draw_board(screen, game, selected_piece, valid_moves, forced_jump_piece)
        
        # Handle AI turn with delay
        if game.current_player == 'W' and not game.game_over() and not ai_thinking:
            ai_thinking = True
            ai_move_timer = pygame.time.get_ticks() + 500  # 500ms delay
        
        # Execute AI move after delay
        if ai_thinking and pygame.time.get_ticks() >= ai_move_timer:
            ai_move(game, agent)
            ai_thinking = False
            
            # Check game over after AI move
            if game.game_over():
                game_over = True
                winner_text = game.get_winner()
        
        # Handle events
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                elif event.key == pygame.K_r and game_over:
                    # Restart game
                    game = Checkers()
                    game_over = False
                    selected_piece = None
                    winner_text = ""
                    ai_thinking = False
            
            elif event.type == pygame.MOUSEBUTTONDOWN and not game_over:
                # Only allow human moves on Black's turn and when AI is not thinking
                if game.current_player == 'B' and not ai_thinking:
                    x, y = pygame.mouse.get_pos()
                    row, col = y // 75, x // 75
                    
                    # Check if click is on a dark square (valid square)
                    if (row + col) % 2 != 0:  # Dark square
                        if selected_piece is None:
                            # Select a piece
                            piece = game.board[row][col]
                            if piece != ' ' and piece[0] == 'B':  # Human's piece
                                # If forced jump, only allow selecting pieces that can jump
                                if game.must_jump:
                                    for jump in game.valid_jump_moves:
                                        if jump[0] == row and jump[1] == col:
                                            selected_piece = (row, col)
                                            break
                                else:
                                    selected_piece = (row, col)
                        else:
                            # Try to make a move
                            from_row, from_col = selected_piece
                            
                            if game.must_jump:
                                # Handle forced jump
                                success = game.forced_jump_move(row, col)
                                if success:
                                    selected_piece = None
                                    # Check game over after move
                                    if game.game_over():
                                        game_over = True
                                        winner_text = game.get_winner()
                                else:
                                    # Invalid jump, try selecting a new piece
                                    piece = game.board[row][col]
                                    if piece != ' ' and piece[0] == 'B':
                                        if game.must_jump:
                                            for jump in game.valid_jump_moves:
                                                if jump[0] == row and jump[1] == col:
                                                    selected_piece = (row, col)
                                                    break
                                        else:
                                            selected_piece = (row, col)
                                    else:
                                        selected_piece = None
                            else:
                                # Handle normal move
                                if game.is_valid_move(from_row, from_col, row, col):
                                    success = game.make_move(from_row, from_col, row, col)
                                    if success:
                                        selected_piece = None
                                        # Check game over after move
                                        if game.game_over():
                                            game_over = True
                                            winner_text = game.get_winner()
                                    else:
                                        selected_piece = None
                                else:
                                    # Try selecting a new piece
                                    piece = game.board[row][col]
                                    if piece != ' ' and piece[0] == 'B':
                                        selected_piece = (row, col)
                                    else:
                                        selected_piece = None
                    else:
                        # Clicked on light square, deselect
                        selected_piece = None
        
        # Check for game over between turns
        if not game_over and game.game_over():
            game_over = True
            winner_text = game.get_winner()
    
    else:
        # Game over screen
        show_game_over_screen(screen, game, winner_text)
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                elif event.key == pygame.K_r:
                    # Restart game
                    game = Checkers()
                    game_over = False
                    selected_piece = None
                    winner_text = ""
                    ai_thinking = False

pygame.quit()

you can use this code to make two agents play each other

#### Agents were modified so they could play both sides.

In [7]:
# main_ai_vs_ai.py
import pygame

# Initialize Pygame
pygame.init()
screen = pygame.display.set_mode((600, 600))
pygame.display.set_caption("Checkers - AI vs AI")

# Load game instance
game = Checkers()

# Choose AI agents (Modify as needed)
# Black plays first, White plays second
ai_player_B = AlphaBetaAgent(depth=5, ai_color= 'B')  # AI controlling Black pieces (goes first)
ai_player_W = ExpectimaxAgent(depth=4)  # AI controlling White pieces (goes second)

# Alternative configurations:
# ai_player_B = MinimaxAgent(depth=4)
# ai_player_W = AlphaBetaAgent(depth=6)
# ai_player_B = ExpectimaxAgent(depth=4)
# ai_player_W = MinimaxAgent(depth=4)

# Colors
LIGHT_BROWN = (205, 133, 63)
DARK_BROWN = (139, 69, 19)
BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
GRAY = (200, 200, 200)
RED = (255, 0, 0)

def draw_board(screen, game):
    """ Draws the Checkers board and pieces """
    # Draw the board
    for row in range(8):
        for col in range(8):
            if (row + col) % 2 == 0:
                color = LIGHT_BROWN
            else:
                color = DARK_BROWN
            pygame.draw.rect(screen, color, (col * 75, row * 75, 75, 75))
    
    # Draw pieces
    for row in range(8):
        for col in range(8):
            piece = game.board[row][col]
            if piece != ' ':
                center_x = col * 75 + 37
                center_y = row * 75 + 37
                
                # Draw piece
                if piece[0] == 'B':
                    color = BLACK
                else:
                    color = WHITE
                pygame.draw.circle(screen, color, (center_x, center_y), 30)
                pygame.draw.circle(screen, GRAY, (center_x, center_y), 30, 2)
                
                # Draw crown for kings
                if len(piece) == 2 and piece[1] == 'K':
                    crown_color = (255, 215, 0)  # Gold
                    pygame.draw.polygon(screen, crown_color, [
                        (center_x - 12, center_y - 8),
                        (center_x - 6, center_y - 14),
                        (center_x, center_y - 8),
                        (center_x + 6, center_y - 14),
                        (center_x + 12, center_y - 8)
                    ])
    
    # Display game info
    font = pygame.font.Font(None, 36)
    small_font = pygame.font.Font(None, 24)
    
    # Turn count
    turn_text = font.render(f"Turn: {game.turn_count}", True, WHITE)
    screen.blit(turn_text, (10, 10))
    
    # Current player
    if game.current_player == 'B':
        player_text = "Black's Turn (Alpha-Beta)"
        player_color = BLACK
    else:
        player_text = "White's Turn (Expectimax)"
        player_color = WHITE
    
    player_surface = font.render(player_text, True, player_color)
    screen.blit(player_surface, (10, 50))
    
    # Piece count
    black_count = sum(row.count('B') + row.count('BK') for row in game.board)
    white_count = sum(row.count('W') + row.count('WK') for row in game.board)
    count_text = small_font.render(f"Black: {black_count}  White: {white_count}", True, WHITE)
    screen.blit(count_text, (10, 90))
    
    # Forced jump indicator
    if game.must_jump:
        jump_text = small_font.render("MUST JUMP!", True, RED)
        screen.blit(jump_text, (10, 115))
    
    pygame.display.flip()

def show_game_over_screen(screen, game, winner_text, stats):
    """Display game over screen with statistics"""
    overlay = pygame.Surface((600, 600))
    overlay.set_alpha(200)
    overlay.fill((0, 0, 0))
    screen.blit(overlay, (0, 0))
    
    font_large = pygame.font.Font(None, 72)
    font_medium = pygame.font.Font(None, 48)
    font_small = pygame.font.Font(None, 36)
    
    # Winner text
    winner_surface = font_large.render(winner_text, True, (255, 215, 0))
    winner_rect = winner_surface.get_rect(center=(300, 150))
    screen.blit(winner_surface, winner_rect)
    
    # Final score
    black_count = sum(row.count('B') + row.count('BK') for row in game.board)
    white_count = sum(row.count('W') + row.count('WK') for row in game.board)
    score_text = font_medium.render(f"Final Score: Black {black_count} - {white_count} White", True, WHITE)
    score_rect = score_text.get_rect(center=(300, 250))
    screen.blit(score_text, score_rect)
    
    # Game statistics
    y_offset = 320
    stats_text = font_small.render(f"Total Turns: {stats['turns']}", True, GRAY)
    stats_rect = stats_text.get_rect(center=(300, y_offset))
    screen.blit(stats_text, stats_rect)
    
    y_offset += 40
    black_moves_text = font_small.render(f"Black Moves Made: {stats['black_moves']}", True, BLACK)
    black_moves_rect = black_moves_text.get_rect(center=(300, y_offset))
    screen.blit(black_moves_text, black_moves_rect)
    
    y_offset += 40
    white_moves_text = font_small.render(f"White Moves Made: {stats['white_moves']}", True, WHITE)
    white_moves_rect = white_moves_text.get_rect(center=(300, y_offset))
    screen.blit(white_moves_text, white_moves_rect)
    
    # Restart instruction
    restart_text = font_small.render("Press R to restart or ESC to quit", True, GRAY)
    restart_rect = restart_text.get_rect(center=(300, 480))
    screen.blit(restart_text, restart_rect)
    
    pygame.display.flip()

def ai_move_with_stats(game, agent, player_color, stats):
    """Execute AI move and update statistics"""
    valid_moves = game.get_valid_moves()
    
    if not valid_moves:
        print(f"No valid moves for {player_color}. Skipping turn.")
        return False
    
    # Get AI move
    ai_move = agent.get_move(game)
    
    if ai_move:
        if len(ai_move) == 4:
            success = game.make_move(ai_move[0], ai_move[1], ai_move[2], ai_move[3])
            if success:
                if player_color == 'B':
                    stats['black_moves'] += 1
                else:
                    stats['white_moves'] += 1
                
                # Handle multiple jumps
                while game.must_jump and game.waiting_for_second_jump:
                    jump_moves = game.valid_jump_moves
                    if jump_moves:
                        jump = jump_moves[0]
                        game.forced_jump_move(jump[2], jump[3])
                        if player_color == 'B':
                            stats['black_moves'] += 1
                        else:
                            stats['white_moves'] += 1
                    else:
                        break
                return True
    return False

# Game variables
running = True
game_over = False
winner_text = ""
stats = {
    'turns': 0,
    'black_moves': 0,
    'white_moves': 0
}

# AI vs AI Game Loop
clock = pygame.time.Clock()

print("=" * 50)
print("Checkers: AI vs AI Match")
print(f"Black AI: {ai_player_B.__class__.__name__} (Depth: {ai_player_B.depth})")
print(f"White AI: {ai_player_W.__class__.__name__} (Depth: {ai_player_W.depth})")
print("=" * 50)

while running:
    pygame.event.pump()  # Prevent freezing
    draw_board(screen, game)
    
    if game.game_over():
        game_over = True
        winner_text = game.get_winner()
        print("\n" + "=" * 50)
        print(f"Game Over! {winner_text}")
        print(f"Final Score - Black: {sum(row.count('B') + row.count('BK') for row in game.board)}")
        print(f"             White: {sum(row.count('W') + row.count('WK') for row in game.board)}")
        print(f"Total Turns: {stats['turns']}")
        print(f"Black Moves: {stats['black_moves']}")
        print(f"White Moves: {stats['white_moves']}")
        print("=" * 50)
        
        # Show game over screen
        show_game_over_screen(screen, game, winner_text, stats)
        pygame.display.flip()
        
        # Wait for user input
        waiting_for_restart = True
        while waiting_for_restart:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                    waiting_for_restart = False
                elif event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_ESCAPE:
                        running = False
                        waiting_for_restart = False
                    elif event.key == pygame.K_r:
                        # Restart game
                        game = Checkers()
                        game_over = False
                        winner_text = ""
                        stats = {
                            'turns': 0,
                            'black_moves': 0,
                            'white_moves': 0
                        }
                        waiting_for_restart = False
                        print("\n" + "=" * 50)
                        print("Game Restarted!")
                        print(f"Black AI: {ai_player_B.__class__.__name__} (Depth: {ai_player_B.depth})")
                        print(f"White AI: {ai_player_W.__class__.__name__} (Depth: {ai_player_W.depth})")
                        print("=" * 50)
        continue
    
    # Check if current player has valid moves
    valid_moves = game.get_valid_moves()
    
    if not valid_moves:
        # If no valid moves, skip the turn
        print(f"No valid moves for {'Black' if game.current_player == 'B' else 'White'}. Skipping turn.")
        game.current_player = 'B' if game.current_player == 'W' else 'W'
        stats['turns'] += 1
        pygame.time.delay(500)
        continue
    
    # Small delay for better visualization
    pygame.time.delay(1000)
    
    # AI makes its move
    if game.current_player == 'B':
        print(f"\nTurn {stats['turns'] + 1}: Black AI thinking...")
        success = ai_move_with_stats(game, ai_player_B, 'B', stats)
        if success:
            print(f"Black made move #{stats['black_moves']}")
    else:
        print(f"\nTurn {stats['turns'] + 1}: White AI thinking...")
        success = ai_move_with_stats(game, ai_player_W, 'W', stats)
        if success:
            print(f"White made move #{stats['white_moves']}")
    
    # Update turn count after successful move
    if not game.must_jump:
        stats['turns'] += 1
    
    # Print current scores periodically
    if stats['turns'] % 10 == 0 and stats['turns'] > 0:
        black_count = sum(row.count('B') + row.count('BK') for row in game.board)
        white_count = sum(row.count('W') + row.count('WK') for row in game.board)
        print(f"\n--- After {stats['turns']} turns ---")
        print(f"Score: Black {black_count} - {white_count} White")
        print(f"Black Kings: {sum(row.count('BK') for row in game.board)}")
        print(f"White Kings: {sum(row.count('WK') for row in game.board)}")
        print("-------------------------\n")
    
    clock.tick(60)

pygame.quit()

Checkers: AI vs AI Match
Black AI: AlphaBetaAgent (Depth: 5)
White AI: ExpectimaxAgent (Depth: 4)

Turn 1: Black AI thinking...
Black made move #1

Turn 2: White AI thinking...
White made move #1

Turn 3: Black AI thinking...
Black made move #2

Turn 4: White AI thinking...
White made move #2

Turn 5: Black AI thinking...
Black made move #3

Turn 6: White AI thinking...
White made move #3

Turn 7: Black AI thinking...
Black made move #4

Turn 8: White AI thinking...
White made move #5

Turn 9: Black AI thinking...
Black made move #5

Turn 10: White AI thinking...
White made move #6

--- After 10 turns ---
Score: Black 9 - 9 White
Black Kings: 0
White Kings: 0
-------------------------


Turn 11: Black AI thinking...
Black made move #6

Turn 12: White AI thinking...
White made move #8

Turn 13: Black AI thinking...
Black made move #7

Turn 14: White AI thinking...
White made move #9

Turn 15: Black AI thinking...
Black made move #8

Turn 16: White AI thinking...
White made move #10

Tur